# Model Governance Notebook

This notebook compares feature-set candidates with apples-to-apples replay outputs.

Primary decision metric:
- `expected_k_mae_on_matched`

Guardrails:
- `k_rate_mae_on_matched`
- calibration (`brier`, `logloss`, `ece`)
- money (`roi`, `clv_mean_pp`)
- risk (`sortino`, `max_drawdown_abs`, `calmar`, `profit_factor`, `cvar_95`, `expectancy_per_bet`, `turnover_stability`)

In [ ]:
from pathlib import Path
import polars as pl
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "production").exists() and (candidate / "src" / "Python").exists():
        ROOT = candidate
        break

ODDS_DIR = ROOT / "artifacts" / "odds_log"
REPLAY_PATH = ODDS_DIR / "feature_set_governance_compare.csv"

if not REPLAY_PATH.exists():
    raise FileNotFoundError(
        f"Missing {REPLAY_PATH}. Run production/ops/compare_feature_set_governance.py first."
    )

df = pl.read_csv(REPLAY_PATH)
df

In [ ]:
core_cols = [
    "feature_set",
    "calibration_mode",
    "expected_k_mae_on_matched",
    "k_rate_mae_on_matched",
    "expected_k_mae_tbf_minus_3pct",
    "expected_k_mae_tbf_plus_3pct",
    "tbf_sensitivity_mae_delta",
    "brier",
    "logloss",
    "ece",
    "mce",
    "brier_skill_vs_market",
    "logloss_skill_vs_market",
    "roi",
    "clv_mean_pp",
    "sortino",
    "max_drawdown_abs",
    "max_drawdown_pct",
    "max_recovery_bets",
    "calmar",
    "profit_factor",
    "cvar_95",
    "expectancy_per_bet",
    "turnover_stability",
    "go_no_go_replay_ci_gate",
    "n_policy_bets",
]

ranked = (
    df.select([c for c in core_cols if c in df.columns])
    .sort(
        ["go_no_go_replay_ci_gate", "expected_k_mae_on_matched", "tbf_sensitivity_mae_delta", "roi"],
        descending=[True, False, False, True],
    )
)
ranked

In [ ]:
# Calibration-mode parity view per feature set
pivot = (
    ranked
    .select([
        "feature_set",
        "calibration_mode",
        "expected_k_mae_on_matched",
        "k_rate_mae_on_matched",
        "brier",
        "logloss",
        "ece",
        "roi",
        "sortino",
    ])
    .pivot(
        values=[
            "expected_k_mae_on_matched",
            "k_rate_mae_on_matched",
            "brier",
            "logloss",
            "ece",
            "roi",
            "sortino",
        ],
        index="feature_set",
        columns="calibration_mode",
    )
)
pivot

In [ ]:
# Rolling stability proxy: daily policy turnover dispersion
if "n_policy_bets" in df.columns:
    best = ranked.head(1)
    print("Top candidate row:")
    print(best)

# Optional quick scan chart: expected_K MAE vs ROI
pdf = ranked.to_pandas()
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(
    pdf["expected_k_mae_on_matched"],
    pdf["roi"],
    alpha=0.8,
)
ax.set_xlabel("expected_K MAE (lower better)")
ax.set_ylabel("ROI (higher better)")
ax.set_title("Candidate Tradeoff: Accuracy vs ROI")
ax.grid(alpha=0.2)
plt.show()

In [ ]:
RANK_PATH = ODDS_DIR / "feature_set_governance_ranked.csv"
DAILY_PATH = ODDS_DIR / "feature_set_governance_daily.csv"
SEG_PATH = ODDS_DIR / "feature_set_governance_segments.csv"

ranked_governance = pl.read_csv(RANK_PATH) if RANK_PATH.exists() else pl.DataFrame()
daily_governance = pl.read_csv(DAILY_PATH) if DAILY_PATH.exists() else pl.DataFrame()
segment_governance = pl.read_csv(SEG_PATH) if SEG_PATH.exists() else pl.DataFrame()

print("ranked rows:", ranked_governance.height)
print("daily rows:", daily_governance.height)
print("segment rows:", segment_governance.height)
ranked_governance.head(15)

In [ ]:
# Regime stability snapshot (side / market / odds / line bands)
segment_summary = (
    segment_governance
    .group_by(["feature_set", "calibration_mode", "segment_type"])
    .agg([
        pl.col("roi").mean().alias("roi_mean_across_segments"),
        pl.col("roi").std().alias("roi_std_across_segments"),
        pl.col("n_bets").sum().alias("n_bets_total"),
    ])
    .with_columns(
        pl.when(pl.col("roi_mean_across_segments").abs() > 1e-9)
        .then(pl.col("roi_std_across_segments") / pl.col("roi_mean_across_segments").abs())
        .otherwise(None)
        .alias("roi_dispersion_ratio")
    )
    .sort(["feature_set", "calibration_mode", "segment_type"])
)
segment_summary

In [ ]:
# Freeze-vs-continue decision frame
must_cols = [
    "feature_set",
    "calibration_mode",
    "gate_pass_count",
    "expected_k_mae_on_matched",
    "k_rate_mae_on_matched",
    "brier",
    "ece",
    "roi",
    "sortino",
    "profit_factor",
    "tbf_sensitivity_mae_delta",
]

view = ranked_governance.select([c for c in must_cols if c in ranked_governance.columns]).head(20)
view

if view.height > 0:
    top = view.row(0, named=True)
    print("\nTop current candidate:")
    print(top)
    if top.get("gate_pass_count", 0) >= 5:
        print("\nPreliminary verdict: close to freeze-quality; run final segment and CI checks before hard freeze.")
    else:
        print("\nPreliminary verdict: continue search; candidate does not clear enough governance gates yet.")

In [ ]:
# Calibration quality extension: MCE + skill-vs-market view
calib_cols = [
    "feature_set",
    "calibration_mode",
    "n_policy_bets",
    "brier",
    "logloss",
    "ece",
    "mce",
    "brier_skill_vs_market",
    "logloss_skill_vs_market",
]

calib_view = (
    ranked_governance.select([c for c in calib_cols if c in ranked_governance.columns])
    .sort(["brier_skill_vs_market", "logloss_skill_vs_market", "ece"], descending=[True, True, False])
)
calib_view.head(20)

In [ ]:
# Edge-decile lift: does higher edge realize better ROI/CLV?
EDGE_DECILE_PATH = ODDS_DIR / "feature_set_governance_edge_deciles.csv"
edge_dec = pl.read_csv(EDGE_DECILE_PATH) if EDGE_DECILE_PATH.exists() else pl.DataFrame()

if edge_dec.is_empty():
    print(f"Missing {EDGE_DECILE_PATH}. Re-run compare_feature_set_governance.py")
else:
    top = ranked_governance.sort(
        ["gate_pass_count", "composite_score"],
        descending=[True, True],
    ).head(1)
    if top.height == 0:
        print("No ranked governance rows available.")
    else:
        top_row = top.row(0, named=True)
        fs = str(top_row.get("feature_set"))
        cm = str(top_row.get("calibration_mode"))
        lift = edge_dec.filter(
            (pl.col("feature_set") == fs) & (pl.col("calibration_mode") == cm)
        ).sort("edge_decile")

        print(f"Top candidate: {fs} [{cm}]")
        display_cols = [
            "edge_decile",
            "n_bets",
            "mean_edge",
            "roi",
            "mean_clv_pp",
        ]
        print(lift.select([c for c in display_cols if c in lift.columns]))

        if lift.height > 0:
            pdf_lift = lift.to_pandas()
            fig, ax = plt.subplots(figsize=(9, 4.5))
            ax.plot(pdf_lift["edge_decile"], pdf_lift["roi"], marker="o", label="ROI")
            ax.axhline(0, color="gray", linestyle="--", linewidth=1)
            ax.set_xlabel("Edge Decile (1=lowest, 10=highest)")
            ax.set_ylabel("ROI")
            ax.set_title("Edge-Decile ROI Lift (Top Candidate)")
            ax.grid(alpha=0.2)
            plt.show()

            fig, ax = plt.subplots(figsize=(9, 4.5))
            ax.plot(pdf_lift["edge_decile"], pdf_lift["mean_clv_pp"], marker="o", label="Mean CLV pp")
            ax.axhline(0, color="gray", linestyle="--", linewidth=1)
            ax.set_xlabel("Edge Decile (1=lowest, 10=highest)")
            ax.set_ylabel("Mean CLV (pp)")
            ax.set_title("Edge-Decile CLV Lift (Top Candidate)")
            ax.grid(alpha=0.2)
            plt.show()

In [ ]:
# Drawdown and recovery quality panel
risk_cols = [
    "feature_set",
    "calibration_mode",
    "n_policy_bets",
    "roi",
    "sortino",
    "max_drawdown_abs",
    "max_drawdown_pct",
    "max_recovery_bets",
    "cvar_95",
    "profit_factor",
]

risk_view = ranked_governance.select([c for c in risk_cols if c in ranked_governance.columns])

# Lower drawdown/recovery is better; higher sortino/profit factor is better.
risk_ranked = risk_view.sort(
    ["max_drawdown_pct", "max_recovery_bets", "sortino", "profit_factor"],
    descending=[False, False, True, True],
)
risk_ranked.head(20)